# Networks to test

In [ ]:
import torch
device = "cuda"

In [ ]:
# Test forward pass with a dummy input
def test_forward_pass(model, dimensions=(1, 1, 96, 96, 96)):
    print("\n--- Testing Forward Pass ---")
    
    # Create a dummy input tensor with the specified dimensions
    dummy_input = torch.randn(*dimensions).to(device)  # e.g., (batch_size, in_channels, D, H, W)

    # Perform the forward pass (within a no_grad context for efficiency during testing)
    with torch.no_grad():
        try:
            output = model(dummy_input)
            print("Forward pass successful!")
            print(f"Output shape: {output.shape}")
            expected_shape = (dimensions[0], 1, dimensions[2], dimensions[3], dimensions[4])
            assert output.shape == expected_shape, f"Expected output shape {expected_shape}, but got {output.shape}"
        except Exception as e:
            print(f"Error during forward pass: {e}")


## Swin UNETR
* Data availability: Transformer-based models like Swin UNETR can perform effectively even with limited labeled data, making them suitable for scenarios with smaller datasets.

In [ ]:
import torch
from monai.networks.nets import SwinUNETR
from os.path import join

def fix_checkpoint_keys(state_dict):
    """
    This function replaces the 'module.' with 'swinViT.' 
    and the linear layers '.mlp.fc' with '.mlp.linear'
    """
    fixed_state_dict = {}
    for key, value in state_dict.items():
        new_key = key
        if key.startswith("module."):
            new_key = key.replace('module.', 'swinViT.')  # Replace module by swinViT
            new_key = new_key.replace('.mlp.fc', '.mlp.linear')  # Replace module by swinViT
             
        fixed_state_dict[new_key] = value
    return fixed_state_dict

def load_pretrained_swinvit(ckpt_path, img_size=(96, 96, 96), in_channels=1, out_channels=14, feature_size=48, use_checkpoint=True, verbose=False):
    """
    Load a pretrained SwinViT model from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint file.
        img_size (tuple): Image size for the model.
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        feature_size (int): Feature size for the model.
        use_checkpoint (bool): Whether to use checkpointing.

    Returns:
        model: The loaded SwinUNETR model.
    """

    # Load the checkpoint
    model_dict = torch.load(ckpt_path, weights_only=False)
    state_dict = model_dict.get("state_dict", model_dict) 

    # Fix keys in the checkpoint
    state_dict = fix_checkpoint_keys(state_dict)

    # Initialize the model
    model = SwinUNETR(
        img_size=img_size,
        in_channels=in_channels,
        out_channels=out_channels,
        feature_size=feature_size,
        use_checkpoint=use_checkpoint,
    )

    if verbose:
        # Check for mismatches
        model_keys = set(model.state_dict().keys())
        checkpoint_keys = set(state_dict.keys())

        # Check matching and mismatching keys
        loaded_keys = model_keys & checkpoint_keys
        not_loaded_keys = model_keys - checkpoint_keys
        extra_keys_in_checkpoint = checkpoint_keys - model_keys

        print(f"\n✅ Loaded {len(loaded_keys)} keys:")
        for k in sorted(loaded_keys):
            print(f"  {k}")

        print(f"\n❌ Not Loaded ({len(not_loaded_keys)}):")
        for k in sorted(not_loaded_keys):
            print(f"  {k}")

        print(f"\n📦 Extra keys in checkpoint (ignored): {len(extra_keys_in_checkpoint)}")
        for k in sorted(extra_keys_in_checkpoint):
            print(f"  {k}")

    # Load the state_dict into the model
    model.load_state_dict(state_dict, strict=False)
    print("✅ Model weights loaded (non-strict mode).")
    return model


ckpt_path = join("/projects/nian/synthrad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model/network/pre_trained/SwinUNETR/model_swinvit.pt")
model = load_pretrained_swinvit(ckpt_path, verbose=True, )

In [ ]:
# Load the model trained on the BTCV multi-organ dataset
from os.path import join

def load_pretrained_SwinUNETR(ckpt_path, img_size=(96, 96, 96), in_channels=1, out_channels=14, feature_size=48, use_checkpoint=True, verbose=False):
    """
    Load a pretrained SwinUNETR model from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint file.
        img_size (tuple): Image size for the model.
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        feature_size (int): Feature size for the model.
        use_checkpoint (bool): Whether to use checkpointing.

    Returns:
        model: The loaded SwinUNETR model.
    """
    # Load the checkpoint
    model_dict = torch.load(ckpt_path, weights_only=False)

    # Initialize the model
    model = SwinUNETR(
        img_size=img_size,
        in_channels=in_channels,
        out_channels=out_channels,
        feature_size=feature_size,
        use_checkpoint=use_checkpoint,
    )

    # Extract state_dict from the checkpoint
    state_dict = model_dict.get("state_dict", model_dict)
    if verbose:
        # Check for mismatches
        model_keys = set(model.state_dict().keys())
        checkpoint_keys = set(state_dict.keys())

        # Check matching and mismatching keys
        loaded_keys = model_keys & checkpoint_keys
        not_loaded_keys = model_keys - checkpoint_keys
        extra_keys_in_checkpoint = checkpoint_keys - model_keys

        print(f"\n✅ Loaded {len(loaded_keys)} keys:")
        for k in sorted(loaded_keys):
            print(f"  {k}")

        print(f"\n❌ Not Loaded ({len(not_loaded_keys)}):")
        for k in sorted(not_loaded_keys):
            print(f"  {k}")

        print(f"\n📦 Extra keys in checkpoint (ignored): {len(extra_keys_in_checkpoint)}")
        for k in sorted(extra_keys_in_checkpoint):
            print(f"  {k}")

    # Load the state_dict into the model
    model.load_state_dict(state_dict)
    print("✅ Model weights loaded (strict mode).")
    return model


ckpt_path = join('/projects/nian/synthrad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model/network/pre_trained/SwinUNETR/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt')
model = load_pretrained_SwinUNETR(ckpt_path, verbose=True)

## Basic U-Net

In [ ]:
from monai.networks.nets import UNet
net = UNet(
    spatial_dims=3,
    in_channels=2,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

test_forward_pass(model, dimensions=(1, 2, 96, 96, 96))

### Total Segmentator based nnUNet

In [ ]:
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
import nnunetv2
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
from batchgenerators.utilities.file_and_folder_operations import load_json, join, isfile, maybe_mkdir_p, isdir, subdirs, \
    save_json
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager, ConfigurationManager
from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels

In [ ]:
def load_pretrained_TotalSegmentator(ckpt_path, verbose):
    """
    Load a pretrained TotalSegmentator model from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint directory.
        verbose (bool): Whether to print detailed information about the loading process.

    Returns:
        model: The loaded TotalSegmentator model.
    """
    checkpoint_name = 'checkpoint_final.pth'

    # Load dataset and plans JSON files
    dataset_json = load_json(join(ckpt_path, 'dataset.json'))
    plans = load_json(join(ckpt_path, 'plans.json'))
    plans_manager = PlansManager(plans)

    # Define folds to use
    use_folds = [0] # Always use fold 0
    if isinstance(use_folds, str):
        use_folds = [use_folds]

    # Load network weights from checkpoints
    parameters = []
    for i, f in enumerate(use_folds):
        f = int(f) if f != 'all' else f
        checkpoint = torch.load(
            join(ckpt_path, f'fold_{f}', checkpoint_name),
            map_location=torch.device('cpu'),
            weights_only=False
        )
        if i == 0:
            trainer_name = checkpoint['trainer_name']
            configuration_name = checkpoint['init_args']['configuration']
            inference_allowed_mirroring_axes = checkpoint.get('inference_allowed_mirroring_axes', None)

        parameters.append(checkpoint['network_weights'])

    # Get configuration manager and input channels
    configuration_manager = plans_manager.get_configuration(configuration_name)
    num_input_channels = determine_num_input_channels(plans_manager, configuration_manager, dataset_json)

    # Find the trainer class
    trainer_class = recursive_find_python_class(
        join(nnunetv2.__path__[0], "training", "nnUNetTrainer"),
        trainer_name,
        'nnunetv2.training.nnUNetTrainer'
    )

    # Build the network architecture
    model = trainer_class.build_network_architecture(
        configuration_manager.network_arch_class_name,
        configuration_manager.network_arch_init_kwargs,
        configuration_manager.network_arch_init_kwargs_req_import,
        num_input_channels,
        plans_manager.get_label_manager(dataset_json).num_segmentation_heads,
        enable_deep_supervision=False
    )

    # Verbose output for key mismatches
    if verbose:
        model_keys = set(model.state_dict().keys())
        checkpoint_keys = set(parameters[0].keys())

        loaded_keys = model_keys & checkpoint_keys
        not_loaded_keys = model_keys - checkpoint_keys
        extra_keys_in_checkpoint = checkpoint_keys - model_keys

        print(f"\n✅ Loaded {len(loaded_keys)} keys:")
        for k in sorted(loaded_keys):
            print(f"  {k}")

        print(f"\n❌ Not Loaded ({len(not_loaded_keys)}):")
        for k in sorted(not_loaded_keys):
            print(f"  {k}")

        print(f"\n📦 Extra keys in checkpoint (ignored): {len(extra_keys_in_checkpoint)}")
        for k in sorted(extra_keys_in_checkpoint):
            print(f"  {k}")

    # Load the state_dict into the model
    model.load_state_dict(parameters[0])
    print("✅ Model weights loaded (strict mode).")
    return model

ckpt_path = '/projects/nian/synthrad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model/network/pre_trained/TotalSegmentator/Dataset297_TotalSegmentator_total_3mm_1559subj/nnUNetTrainer_4000epochs_NoMirroring__nnUNetPlans__3d_fullres'
model = load_pretrained_TotalSegmentator(ckpt_path, verbose=False)
    

# Apply new losses



In [ ]:
import os
import torch
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
import nnunetv2
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
from batchgenerators.utilities.file_and_folder_operations import load_json, join
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels
def load_pretrained_TotalSegmentator(ckpt_path):
    """
    Load a pretrained TotalSegmentator model from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint directory.
        verbose (bool): Whether to print detailed information about the loading process.

    Returns:
        model: The loaded TotalSegmentator model.
    """
    os.environ['nnUNet_raw'] = ''
    os.environ['nnUNet_preprocessed'] = ''
    os.environ['nnUNet_results'] = ''
    checkpoint_name = 'checkpoint_final.pth'
    try:
        ckpt_path = ckpt_path.split('/fold_0')[0]
    except:
        raise Exception('The checkpoint_final.pth needs to be inside of the folder fold_0')
    dataset_json = load_json(join(ckpt_path, 'dataset.json'))
    plans = load_json(join(ckpt_path, 'plans.json'))
    plans_manager = PlansManager(plans)
    use_folds = [0]
    if isinstance(use_folds, str):
        use_folds = [use_folds]
    parameters = []
    for i, f in enumerate(use_folds):
        f = int(f) if f != 'all' else f
        checkpoint = torch.load(join(ckpt_path, f'fold_{f}', checkpoint_name), map_location=torch.device('cpu'), weights_only=False)
        if i == 0:
            trainer_name = checkpoint['trainer_name']
            configuration_name = checkpoint['init_args']['configuration']
            inference_allowed_mirroring_axes = checkpoint.get('inference_allowed_mirroring_axes', None)
        parameters.append(checkpoint['network_weights'])

    configuration_manager = plans_manager.get_configuration(configuration_name)
    num_input_channels = determine_num_input_channels(plans_manager, configuration_manager, dataset_json)
    trainer_class = recursive_find_python_class(join(nnunetv2.__path__[0], 'training', 'nnUNetTrainer'), trainer_name, 'nnunetv2.training.nnUNetTrainer')
    model = trainer_class.build_network_architecture(configuration_manager.network_arch_class_name, configuration_manager.network_arch_init_kwargs, configuration_manager.network_arch_init_kwargs_req_import, num_input_channels, plans_manager.get_label_manager(dataset_json).num_segmentation_heads, enable_deep_supervision=False)
    
    model.load_state_dict(parameters[0])

    print('✅ Model weights loaded (strict mode).')
    return (model)
seg_model = load_pretrained_TotalSegmentator(ckpt_path="/projects/nian/synthrad2025/src/metrics/evaluation/.totalsegmentator/nnunet/results/Dataset297_TotalSegmentator_total_3mm_1559subj/nnUNetTrainer_4000epochs_NoMirroring__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth")

In [ ]:
from monai.transforms import (
    LoadImaged,
    CropForegroundd,
    RandSpatialCropSamplesd,
    AsDiscreted,
    EnsureTyped,
    EnsureType,
    ScaleIntensityRanged,
    ResampleToMatchd,
    EnsureChannelFirstd,
    GridPatchd,
    CopyItemsd,
    CenterSpatialCropd,
    ResizeWithPadOrCropd,
    SpatialCropd,
    Resize,
    Orientationd,
    ResizeWithPadOrCrop,
    NormalizeIntensityd,
    Resized

)
from monai.data import Dataset, DataLoader,CacheDataset
import numpy as np
import nibabel as nib
import json
import os
import torch
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
import nnunetv2
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
from batchgenerators.utilities.file_and_folder_operations import load_json, join
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels
from monai.data import ITKReader

print(f"Doing NormalizeIntensityd")
a_min_ct= -1024
a_max_ct = 3000
task_name = "Task1"
normalization_stats_path = f"/projects/nian/synthrad2025/Dataset/{task_name}_Train_normalization_stats_{a_min_ct}_{a_max_ct}.json"

with open(normalization_stats_path, "r") as stats_file:
    normalization_stats = json.load(stats_file)
ct_mean = normalization_stats.get("ct_mean")
ct_std = normalization_stats.get("ct_std")
print(f"ct_mean: {ct_mean}")
print(f"ct_std: {ct_std}")
if ct_mean is None or ct_std is None:
    raise ValueError("ct_mean or ct_std not found in the normalization stats file.")

train_transforms_list = [
    LoadImaged(keys=["ct", "mask", "mri", "seg"], image_only=True, reader=ITKReader()),
    EnsureChannelFirstd(keys=["ct", "mask", "mri", "seg"]),
    Orientationd(keys=["ct", "mask", "mri", "seg"], axcodes='RAS'),
    CropForegroundd(keys=["ct", "mask", "mri", "seg"], source_key="mask", allow_smaller=False),  # Crop based on mask
    CopyItemsd(keys=["ct", "mri", "mask", "seg"], times=1, names=["ct_fullres", "mri_fullres", "mask_fullres", "seg_fullres"], allow_missing_keys=False),
    ScaleIntensityRanged( 
        keys=["ct_fullres"],
        a_min=a_min_ct,
        a_max=a_max_ct,
        b_min=a_min_ct,
        b_max=a_max_ct,
        clip=True,
    ),
    NormalizeIntensityd(
        keys=["ct_fullres"],
        subtrahend=ct_mean,
        divisor=ct_std,
        nonzero=False,
        channel_wise=False
        ),
    NormalizeIntensityd(
        keys=["mri_fullres"],
        subtrahend=None,
        divisor=None,
        nonzero=False,
        channel_wise=False
        ),
    ResizeWithPadOrCropd(keys=["ct_fullres", "mri_fullres", "mask_fullres", "seg_fullres"], spatial_size=(336, 336, 128), mode=["minimum", "minimum", "minimum", "minimum"]), # To ensure the input size is 336, 336, 128,
    CopyItemsd(keys=["seg_fullres"], times=1, names=["seg_downsample"], allow_missing_keys=False),
    Resized(keys=['seg_downsample'], spatial_size=(112, 112, 128), mode=['nearest'])
]



### With spacing from 1x1x3 to 3x3x3 we need input shape of 336, 336, 128 to match the total segmentator 3mm spacing

file_paths = [
    {
        "ct": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/ct.mha",
        "mask": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mask.mha",
        "mri": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mr.mha",
        "seg": "/projects/nian/synthrad2025/Dataset/Task1_seg/Task1/AB/1ABA005/pred_seg.mha",
    },
    {
        "ct": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/ct.mha",
        "mask": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mask.mha",
        "mri": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mr.mha",
        "seg": "/projects/nian/synthrad2025/Dataset/Task1_seg/Task1/AB/1ABA005/pred_seg.mha",
    },
    {
        "ct": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/ct.mha",
        "mask": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mask.mha",
        "mri": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mr.mha",
        "seg": "/projects/nian/synthrad2025/Dataset/Task1_seg/Task1/AB/1ABA005/pred_seg.mha",
    },
    {
        "ct": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/ct.mha",
        "mask": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mask.mha",
        "mri": "/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/mr.mha",
        "seg": "/projects/nian/synthrad2025/Dataset/Task1_seg/Task1/AB/1ABA005/pred_seg.mha",
    }
]
# Load dataset
dataset = CacheDataset(data=file_paths, cache_rate=1, transform=train_transforms_list)
dataloader = DataLoader(dataset, batch_size=4)  # Process one patient at a time

In [ ]:
import sys
sys.path.append("/projects/nian/synthrad2025/src/metrics/functions")
sys.path.append("/projects/nian/synthrad2025/src/metrics/evaluation")
sys.path.append("/projects/nian/synthrad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model")
sys.path.append("/projects/nian/synthrad2025/src/Synthetic-CT-generation-from-MRI-using-3D-transformer-based-denoising-diffusion-model/network")

from network.Diffusion_model_transformer import *
num_channels=64
attention_resolutions="32,16,8"
channel_mult = (1, 2, 3, 4)
num_heads=[4,4,8,16]
window_size = [[4,4,4],[4,4,4],[4,4,2],[4,4,2]]
num_res_blocks = [2,2,2,2]
sample_kernel=([2,2,2],[2,2,1],[2,2,1],[2,2,1]),
use_scale_shift_norm = True
resblock_updown = False
dropout = 0
use_checkpoint=False

attention_ds = []
for res in attention_resolutions.split(","):
    attention_ds.append(int(res))

A_to_B_model = SwinVITModel(
        image_size=(128, 128, 32),
        in_channels=2,
        model_channels=num_channels,
        out_channels=2,
        dims=3,
        sample_kernel = sample_kernel,
        num_res_blocks=num_res_blocks,
        attention_resolutions=tuple(attention_ds),
        dropout=dropout,
        channel_mult=channel_mult,
        num_classes=None,
        use_checkpoint=use_checkpoint,
        use_fp16=False,
        num_heads=num_heads,
        window_size = window_size,
        num_head_channels=64,
        num_heads_upsample=-1,
        use_scale_shift_norm=use_scale_shift_norm,
        resblock_updown=resblock_updown,
        use_new_attention_order=False,
    )

checkpoint = torch.load("/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_128_128_32_region_HN_TH_AB/wandb/latest-run/files/model/A_to_B_model_latest.pt", weights_only=False)
A_to_B_model.load_state_dict(checkpoint['model_state_dict'])
# Retrieve the epoch and best loss
begin_epoch = checkpoint['epoch']
best_loss = checkpoint['best_loss'] 


In [ ]:
resize_to_seg = Resize(
    spatial_size=(112, 112, 128), 
    size_mode='all', 
    mode='trilinear', 
    align_corners=None, 
    anti_aliasing=False, 
    anti_aliasing_sigma=None,
    lazy=False
    )

resize_back_seg = Resize(
    spatial_size=(336, 336, 128), 
    size_mode='all', 
    mode='nearest', 
    align_corners=None, 
    anti_aliasing=False, 
    anti_aliasing_sigma=None,
    lazy=False
    )

AB = [
        2, # kidney right
        3, # kidney left
        5, # liver
        6, # stomach
        *range(10, 14+1), #lungs
        *range(26, 50+1), #vertebrae
        51, #heart
        79, # spinal cord
        *range(92, 115+1), # ribs
        116 #sternum
    ]


def normalize_for_totalseg(image: torch.Tensor) -> torch.Tensor:
    """
    Normalizes a CT image tensor using predefined mean, std, and intensity bounds.
    """
    mean_intensity = -50.38697721419439
    std_intensity = 503.39235619144
    lower_bound = -1004.0
    upper_bound = 1588.0

    image = torch.clamp(image, lower_bound, upper_bound)
    image = image - mean_intensity
    image = image / torch.max(torch.tensor(std_intensity, device=image.device), torch.tensor(1e-8, device=image.device))

    return image

In [ ]:
def predict_seg(fullres_w_synthpatch):
    """
    Performs segmentation on a batch of 3D medical images.

    Args:
        fullres_w_synthpatch (torch.Tensor): Input tensor of shape (B, C, D, H, W) e.g., (2, 1, 128, 128, 128),
            where B is the batch size, and C, D, H, W are channel and spatial dimensions.

    Returns:
        None. (Assumes further processing or storage of results happens externally.)
    """
    batch_preds = []
    for i in range(fullres_w_synthpatch.shape[0]):
        single_image = fullres_w_synthpatch[i]  # Shape: (C, D, H, W)

        resized = resize_to_seg(single_image)

        normed = normalize_for_totalseg(resized).squeeze(0).T

        # Predict segmentation
        pred = seg_model(normed.unsqueeze(0).unsqueeze(0)).squeeze()  # Squeeze all in one go
        seg_pred = torch.softmax(pred, dim=0).argmax(0).T
        batch_preds.append(seg_pred.unsqueeze(0))

    return torch.stack(batch_preds)



In [ ]:
import random

def random_foreground_crop_batch(
    ct_fullres: torch.Tensor,      # (B, C, H, W, D)
    mri_fullres: torch.Tensor,     # (B, C, H, W, D)
    mask: torch.Tensor,            # (B, C, H, W, D)
    crop_size=(128, 128, 32)
):
    """
    Perform random foreground cropping over a batch.
    Returns cropped volumes and coordinates for each item in batch.
    """
    B, C, H, W, D = ct_fullres.shape
    crop_H, crop_W, crop_D = crop_size

    cropped_ct = torch.empty((B, C, crop_H, crop_W, crop_D), dtype=ct_fullres.dtype, device=ct_fullres.device)
    cropped_mri = torch.empty_like(cropped_ct)
    cropped_mask = torch.empty_like(cropped_ct)
    crop_coords = torch.empty((B, 6), dtype=torch.int32, device=ct_fullres.device)

    mask_squeezed = mask[:, 0]  # Assuming C=1 for the mask.

    for b in range(B):
        fg_indices = (mask_squeezed[b] == 1).nonzero(as_tuple=False)
        if fg_indices.size(0) == 0:
            raise ValueError(f"Sample {b} contains no foreground voxels.")

        idx = torch.randint(0, fg_indices.size(0), (1,), device=ct_fullres.device)
        center_y, center_x, center_z = fg_indices[idx].squeeze(0)

        start_y = torch.clamp(center_y - crop_H // 2, min=0, max=H - crop_H)
        start_x = torch.clamp(center_x - crop_W // 2, min=0, max=W - crop_W)
        start_z = torch.clamp(center_z - crop_D // 2, min=0, max=D - crop_D)

        end_y = start_y + crop_H
        end_x = start_x + crop_W
        end_z = start_z + crop_D

        cropped_ct[b] = ct_fullres[b, :, start_y:end_y, start_x:end_x, start_z:end_z]
        cropped_mri[b] = mri_fullres[b, :, start_y:end_y, start_x:end_x, start_z:end_z]
        cropped_mask[b] = mask[b, :, start_y:end_y, start_x:end_x, start_z:end_z]
        crop_coords[b] = torch.tensor([start_y, start_x, start_z, end_y, end_x, end_z], dtype=torch.int32, device=ct_fullres.device)

    return cropped_ct, cropped_mri, cropped_mask, crop_coords

In [ ]:
from monai.metrics import DiceMetric
from monai.losses import DiceLoss

class TotalSegmentatorLoss():
    def __init__(self, ct_std, ct_mean, clip_min_ct, clip_max_ct, path_weights, device='cuda:0'):
        self.ct_std = torch.tensor(ct_std, device=device)
        self.ct_mean = torch.tensor(ct_mean, device=device)
        self.intensity_min = (clip_min_ct - ct_mean) / ct_std
        self.intensity_max = (clip_max_ct - ct_mean) / ct_std

        print(f"intensity_min: {self.intensity_min}")
        print(f"intensity_max: {self.intensity_max}")

        # Normalization value for the TotalSegmentator
        self.TS_mean_intensity = torch.tensor(-50.38697721419439, device=device) # Move to device
        self.TS_std_intensity = torch.tensor(503.39235619144, device=device) # Already on cuda, ensure consistency
        self.TS_lower_bound = torch.tensor(-1004.0, device=device) # Move to device
        self.TS_upper_bound = torch.tensor(1588.0, device=device) # Move to device


        self.epsilon = torch.tensor(1e-8, device=device) # Already on cuda, ensure consistency
        self.device = device # Store device for later use

        num_classes = 117
        weight = torch.zeros(num_classes, device=device)
        classes_to_use = sorted({
            # Kidneys
            2, 3,
            # Organs
            5,   # liver
            6,   # stomach
            15,  # esophagus
            16,  # trachea
            17,  # thyroid
            51,  # heart
            79,  # spinal cord
            90,  # brain
            91,  # skull

            # Lungs (10–14)
            *range(10, 15),

            # Vertebrae (26–50)
            *range(26, 51),

            # Ribs (92–115)
            *range(92, 116),

            # Sternum
            116
        })

        # Set weights for selected classes
        for cls in classes_to_use:
            if cls < num_classes:
                weight[cls] = 1.0
            else:
                print(f"⚠️ Warning: Class {cls} is out of range for num_classes={num_classes}")
        print(f"weight: {weight}")

        # Set the DSC loss from MONAI
        self.loss_DSC = DiceLoss(
            include_background=False, 
            to_onehot_y=True, 
            sigmoid=False, 
            softmax=True, 
            other_act=None,
            squared_pred=False, 
            jaccard=False, 
            reduction='sum', 
            smooth_nr=1e-05, 
            smooth_dr=1e-05, 
            batch=True, 
            weight=weight).to(device)

        self.seg_model = self.__load_TotalSegmentator_model__(ckpt_path=path_weights)
        self.seg_model = self.seg_model.to(device)

        for param in self.seg_model.parameters():
            param.requires_grad = False
        self.seg_model.eval()
    
    def __load_TotalSegmentator_model__(self, ckpt_path):
        """
        Load a pretrained TotalSegmentator model from a checkpoint.

        Args:
            ckpt_path (str): Path to the checkpoint directory.
            verbose (bool): Whether to print detailed information about the loading process.

        Returns:
            model: The loaded TotalSegmentator model.
        """
        os.environ['nnUNet_raw'] = ''
        os.environ['nnUNet_preprocessed'] = ''
        os.environ['nnUNet_results'] = ''
        checkpoint_name = 'checkpoint_final.pth'
        try:
            ckpt_path = ckpt_path.split('/fold_0')[0]
        except:
            raise Exception('The checkpoint_final.pth needs to be inside of the folder fold_0')
        dataset_json = load_json(join(ckpt_path, 'dataset.json'))
        plans = load_json(join(ckpt_path, 'plans.json'))
        plans_manager = PlansManager(plans)
        use_folds = [0]
        if isinstance(use_folds, str):
            use_folds = [use_folds]
        parameters = []
        for i, f in enumerate(use_folds):
            f = int(f) if f != 'all' else f
            checkpoint = torch.load(join(ckpt_path, f'fold_{f}', checkpoint_name), map_location=torch.device(self.device), weights_only=False)
            if i == 0:
                trainer_name = checkpoint['trainer_name']
                configuration_name = checkpoint['init_args']['configuration']
                #inference_allowed_mirroring_axes = checkpoint.get('inference_allowed_mirroring_axes', None)
            parameters.append(checkpoint['network_weights'])

        configuration_manager = plans_manager.get_configuration(configuration_name)
        num_input_channels = determine_num_input_channels(plans_manager, configuration_manager, dataset_json)
        trainer_class = recursive_find_python_class(join(nnunetv2.__path__[0], 'training', 'nnUNetTrainer'), trainer_name, 'nnunetv2.training.nnUNetTrainer')
        model = trainer_class.build_network_architecture(configuration_manager.network_arch_class_name, configuration_manager.network_arch_init_kwargs, configuration_manager.network_arch_init_kwargs_req_import, num_input_channels, plans_manager.get_label_manager(dataset_json).num_segmentation_heads, enable_deep_supervision=False)
        
        model.load_state_dict(parameters[0])

        print('✅ Model weights loaded (strict mode).')
        return model
        
    def __denormalize__(self, predicted_patch):
        """
        Clip and de-normalize the data using the std and mean of the dataset
        """
        #predicted_patch = predicted_patch.clamp(min=self.intensity_min, max=self.intensity_max)
        predicted_patch = predicted_patch * self.ct_std + self.ct_mean
        return predicted_patch

    def __normalize_for_totalseg__(self, image):
        """
        Normalizes a CT image tensor using predefined mean, std, and intensity bounds.
        """
        image = torch.clamp(image, self.TS_lower_bound, self.TS_upper_bound)
        image = image - self.TS_mean_intensity
        image = image / torch.max(self.TS_std_intensity, self.epsilon)
        return image
    
    def __predict_seg__(self, fullres_w_synthpatch):

        """
        Performs segmentation on a batch of 3D medical images.

        Args:
            fullres_w_synthpatch (torch.Tensor): Input tensor of shape (B, C, D, H, W) e.g., (2, 1, 336, 336, 128),
                where B is the batch size, and C, D, H, W are channel and spatial dimensions.

        Returns:
            Logits
        """
        fullres_w_synthpatch = fullres_w_synthpatch.to(self.device) # IMP: Ensure input is on device

        resized = torch.nn.functional.interpolate(
                fullres_w_synthpatch,
                size=(112, 112, 128), # Target D, H, W respectively
                mode='trilinear',
                align_corners=False # Typically recommended to set to False for interpolation
            )
            
        normed = self.__normalize_for_totalseg__(resized).permute(0, 1, 4, 3, 2)
            
        # Predict segmentation
        pred = self.seg_model(normed)  # Squeeze all in one go
        pred = pred.permute(0, 1, 4, 3, 2)

        return  pred

    def __call__(self, real_full_res, predicted_patch, gt_seg, limits):
        # De-normalise -> Almost as real cases
        real_full_res = self.__denormalize__(real_full_res.to(self.device))
        predicted_patch = self.__denormalize__(predicted_patch.to(self.device))

        # Create the input of the TotalSegmentator by paste back the predicted path into the real case.
        fullres_w_synthpatch = real_full_res.clone()
        for sub_limits_idx, sub_limits in enumerate(limits):
            start_y, start_x, start_z, end_y, end_x, end_z = sub_limits
            fullres_w_synthpatch[sub_limits_idx, :, start_y:end_y, start_x:end_x, start_z:end_z] = predicted_patch[sub_limits_idx]

        # Perform segmentation 
        logits = self.__predict_seg__(fullres_w_synthpatch=fullres_w_synthpatch)

        total_loss = torch.tensor(0.0, device=logits.device)  # make sure it’s on the right device
        for sub_limits_idx, sub_limits in enumerate(limits):
            start_y, start_x, start_z, end_y, end_x, end_z = sub_limits
            # Compute dice loss
            total_loss += self.loss_DSC(
                logits[sub_limits_idx:sub_limits_idx + 1, :, start_y//3-1:end_y//3+1, start_x//3-1:end_x//3+1, start_z:end_z], 
                gt_seg[sub_limits_idx:sub_limits_idx + 1, :, start_y//3-1:end_y//3+1, start_x//3-1:end_x//3+1, start_z:end_z]
                )
            
        return total_loss / len(limits)

In [ ]:
DSCLoss = TotalSegmentatorLoss(
        ct_std=ct_std, 
        ct_mean=ct_mean, 
        clip_min_ct=a_min_ct, 
        clip_max_ct=a_max_ct,
        path_weights="/projects/nian/synthrad2025/src/metrics/evaluation/.totalsegmentator/nnunet/results/Dataset297_TotalSegmentator_total_3mm_1559subj/nnUNetTrainer_4000epochs_NoMirroring__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth"
        )

batch_size = 4

In [ ]:
import time
import json

device = "cuda:0"

for i in range(10):
    for batch in dataloader:
        start_total = time.time()

        start = time.time()
        ct_fullres =  batch['ct_fullres'].cuda()
        mri_fullres = batch['mri_fullres'].cuda()
        mask = batch['mask_fullres'].cuda()
        gt_seg_downsample = batch['seg_downsample'].cuda()
        print(f"Time to move tensors to GPU: {time.time() - start:.4f} seconds")
        
        print(f"ct_fullres: {ct_fullres.max()}")
        print(f"ct_fullres: {ct_fullres.min()}")
        start = time.time()
        cropped_ct, cropped_mri, cropped_mask, place = random_foreground_crop_batch(
            ct_fullres=ct_fullres,
            mri_fullres=mri_fullres,
            mask=mask,
            crop_size=(128, 128, 32)
        )
        print(f"Time for random_foreground_crop_batch: {time.time() - start:.4f} seconds")

        start = time.time()
        fullres_w_synthpatch = ct_fullres.clone()
        print(f"Time to clone ct_fullres: {time.time() - start:.4f} seconds")

        a_min_ct = -1024
        a_max_ct = 3000
        task_name = "Task1"
        normalization_stats_path = f"/projects/nian/synthrad2025/Dataset/{task_name}_Train_normalization_stats_{a_min_ct}_{a_max_ct}.json"

        start = time.time()
        with open(normalization_stats_path, "r") as stats_file:
            normalization_stats = json.load(stats_file)
        ct_mean = normalization_stats.get("ct_mean")
        ct_std = normalization_stats.get("ct_std")
        print(f"Time to load normalization stats: {time.time() - start:.4f} seconds")

        start = time.time()
        dsc_loss = DSCLoss(
            real_full_res=ct_fullres,
            predicted_patch=cropped_ct,
            gt_seg=gt_seg_downsample,
            limits=place
        )
        print(f"Time to compute DSCLoss: {time.time() - start:.4f} seconds")

        print(f"dsc_loss: {dsc_loss}")
        print(f"Total time for one batch: {time.time() - start_total:.4f} seconds\n")
    print("###############")
    


In [ ]:
DSC= tensor([[0.9513]], device='cuda:0')
None
loss: 0.11058387905359268
# dsc_loss: 0.09746934473514557

In [ ]:
import SimpleITK as sitk
for batch in dataloader:
    ct_fullres =  batch['ct_fullres']
    mri_fullres = batch['mri_fullres']
    mask = batch['mask_fullres']

    print(f"ct_fullres: {ct_fullres.shape}")
    print(f"ct_fullres.max: {ct_fullres.max()}")
    print(f"ct_fullres.min: {ct_fullres.min()}")

    print(f"mri_fullres: {mri_fullres.shape}")
    print(f"mri_fullres.max: {mri_fullres.max()}")
    print(f"mri_fullres.min: {mri_fullres.min()}")

    print(f"mask: {mask.shape}")
    print(f"mask.max: {mask.max()}")
    print(f"mask.min: {mask.min()}")

    cropped_ct, cropped_mri, cropped_mask, place = random_foreground_crop(
        ct_fullres=ct_fullres, 
        mri_fullres=mri_fullres, 
        mask=mask, 
        crop_size=(128, 128, 32)
    )
    start_y, start_x, start_z, end_y, end_x, end_z = place
    #####################
    #### TODO remove ####
    nifti_img = nib.Nifti1Image(cropped_ct[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/projects/nian/synthrad2025/trash/cropped_ct.nii.gz")

    nifti_img = nib.Nifti1Image(cropped_mri[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/projects/nian/synthrad2025/trash/cropped_mri.nii.gz")

    nifti_img = nib.Nifti1Image(cropped_mask[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/projects/nian/synthrad2025/trash/cropped_mask.nii.gz")
    ####################

    fullres_w_synthpatch = ct_fullres.clone()
    # De-normlaise
    a_min_ct= -1024
    a_max_ct = 3000
    task_name = "Task1"
    normalization_stats_path = f"/projects/nian/synthrad2025/Dataset/{task_name}_Train_normalization_stats_{a_min_ct}_{a_max_ct}.json"

    with open(normalization_stats_path, "r") as stats_file:
        normalization_stats = json.load(stats_file)
    ct_mean = normalization_stats.get("ct_mean")
    ct_std = normalization_stats.get("ct_std")

    fullres_w_synthpatch  = fullres_w_synthpatch * ct_std + ct_mean # De-normalise the entire volume
    cropped_ct = cropped_ct * ct_std + ct_mean # De-normalise the prediction
    fullres_w_synthpatch[:, :, start_y:end_y, start_x:end_x, start_z:end_z] = cropped_ct

    #####################
    #### TODO remove ####
    # Load the MHA image
    mha_img = sitk.ReadImage("/projects/nian/synthrad2025/Dataset/synthRAD2025_Task1_Train/Task1/AB/1ABA005/ct.mha")

    # Extract components
    origin = np.array(mha_img.GetOrigin())            # (x, y, z)
    spacing = np.array(mha_img.GetSpacing())          # (sx, sy, sz)
    direction = np.array(mha_img.GetDirection())      # flat list of 9 values for 3D
    direction = direction.reshape((3, 3))              # 3x3 matrix

    # Build 4x4 affine matrix
    affine = np.eye(4)
    affine[:3, :3] = direction * spacing[np.newaxis, :]  # scale direction cosines by spacing
    affine[:3, 3] = origin
    

    nifti_img = nib.Nifti1Image(ct_fullres[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/projects/nian/synthrad2025/trash/ct_fullres.nii.gz")

    nifti_img = nib.Nifti1Image(fullres_w_synthpatch[0][0].numpy(), affine=affine)
    nib.save(nifti_img, "/projects/nian/synthrad2025/trash/fullres_w_synthpatch.nii.gz")

    if torch.equal(fullres_w_synthpatch, batch['ct_fullres']):
        print("The pasted back image is equal to the original fullres image.")
    else:
        print("The pasted back image is NOT equal to the original fullres image.")
    ####################

    seg_preds = predict_seg(fullres_w_synthpatch)

    print(f"seg_preds: {seg_preds.shape}")
    for seg_pred in seg_preds:
        seg_pred_resized = resize_back_seg(seg_pred)
        print(f"seg_pred_resized: {seg_pred_resized.shape}")

    #####################
    #### TODO remove ####
    nifti_img = nib.Nifti1Image(seg_preds[0][0].detach().cpu().numpy().astype(np.int16), affine=np.eye(4))
    nib.save(nifti_img, f"/projects/nian/synthrad2025/trash/seg.nii.gz")

    nifti_img = nib.Nifti1Image(seg_pred_resized[0].detach().cpu().numpy().astype(np.int16), affine=affine)
    nib.save(nifti_img, f"/projects/nian/synthrad2025/trash/seg_pred_resized.nii.gz")
    ####################

In [ ]:
for batch in dataloader:
    roi_center = [100, 100, 100]

    print(f"ct_fullres: {batch['ct_fullres'].shape}")
    print(f"ct: {batch['ct'].shape}")
    print(f"mask: {batch['mask'].shape}")
    print(f"mri: {batch['mri'].shape}")

    # Get the fullres CT and MR
    # Paste lowres into output
    fullres_w_synthpatch = ct_fullres.clone()
    fullres_w_synthpatch[:, :, start_z:end_z, start_y:end_y, start_x:end_x] = ct

    # Perform the predication segmentation
    seg_pred = predict_seg(fullres_w_synthpatch)
    print(f"seg_pred: {seg_pred.shape}")

    # Save a sample to NIfTI if needed
    nifti_img = nib.Nifti1Image(seg_pred[0][0].detach().cpu().numpy().astype(np.int16), affine=np.eye(4))
    nib.save(nifti_img, f"/projects/nian/synthrad2025/trash/seg.nii.gz")
    
    nifti_img = nib.Nifti1Image(fullres_w_synthpatch[0][0].numpy(), affine=np.eye(4))
    nib.save(nifti_img, "/projects/nian/synthrad2025/trash/fullres_w_synthpatch.nii.gz")
    
    

    if torch.equal(fullres_w_synthpatch, batch['ct_fullres']):
        print("The pasted back image is equal to the original fullres image.")
    else:
        print("The pasted back image is NOT equal to the original fullres image.")


